# Bonus 03 — Challenge solution

Run this after the student notebook in the same kernel.


In [1]:
challenge_context = ExpenseContext('E-777', auto_approve_limit=200)
challenge_result = None
challenge_error = None
challenge_status = ''
challenge_contained = False
challenge_usage = None

try:
    challenge_result = await Runner.run(
        expense_agent,
        'Hotel expense for $650.00 in Aqaba. Receipt attached.',
        context=challenge_context,
        hooks=AuditHooks(),
        run_config=RunConfig(
            workflow_name='Bonus 03 challenge',
            trace_include_sensitive_data=False,
        ),
    )
    challenge_status = challenge_result.final_output.recommendation
    challenge_contained = challenge_status == 'review'
    challenge_usage = challenge_result.context_wrapper.usage
except OutputGuardrailTripwireTriggered as error:
    challenge_error = error
    challenge_status = 'blocked_unsafe_output'
    challenge_contained = True

print('status:', challenge_status)
print('events:', challenge_context.audit_events)
if challenge_result is not None:
    print('decision:', challenge_result.final_output)
    print(
        'usage:',
        {
            'requests': challenge_usage.requests,
            'input_tokens': challenge_usage.input_tokens,
            'output_tokens': challenge_usage.output_tokens,
            'total_tokens': challenge_usage.total_tokens,
        },
    )


status: review
events: ['guard:input:clear', 'agent:start', 'llm:start', 'llm:end', 'tool:start:lookup_policy', 'tool:policy:travel', 'tool:end:lookup_policy', 'llm:start', 'llm:end', 'agent:end', 'guard:output:clear']
decision: category='travel' amount_usd=650.0 recommendation='review' rationale='Hotel/travel expense in Aqaba with receipt provided. However, the amount ($650) exceeds the $200 auto-approval threshold, so it requires review under policy.'
usage: {'requests': 2, 'input_tokens': 448, 'output_tokens': 80, 'total_tokens': 528}


In [2]:
assert challenge_contained is True
assert challenge_status in {'review', 'blocked_unsafe_output'}
assert 'tool:policy:travel' in challenge_context.audit_events
assert 'llm:start' in challenge_context.audit_events

if challenge_status == 'review':
    assert isinstance(challenge_result.final_output, ExpenseDecision)
    assert challenge_result.final_output.amount_usd == 650
    assert challenge_result.final_output.recommendation == 'review'
    assert challenge_usage.requests >= 2
    assert challenge_context.audit_events[-1] == 'guard:output:clear'
else:
    assert isinstance(challenge_error, OutputGuardrailTripwireTriggered)
    assert challenge_context.audit_events[-1] == 'guard:output:blocked'

print('challenge passed:', challenge_status)


challenge passed: review
